In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from mesa import Agent, Model
import os, webbrowser

# ═══════════════════════════════════════════════════════════════
#  1. LOAD DATA
# ═══════════════════════════════════════════════════════════════
CSV_PATH = r"C:\Users\Dev Sharma\OneDrive\Documents\MDP\Datasets\Combined_Aircraft_Tracking.csv"
raw = pd.read_csv(CSV_PATH)
raw[['lat', 'lon']] = raw['Position'].str.split(',', expand=True).astype(float)
raw['UTC'] = pd.to_datetime(raw['UTC'])
raw = raw.sort_values(['Callsign', 'UTC']).reset_index(drop=True)

# Split into one DataFrame per aircraft
aircraft_data = {cs: g.reset_index(drop=True) for cs, g in raw.groupby('Callsign')}
print(f"Loaded {len(aircraft_data)} aircraft: {list(aircraft_data.keys())}")

# Colour palette — one per aircraft
COLOURS = {
    'AAL1189': '#00c8e0',   # cyan
    'NKS2061': '#ff9d00',   # amber
    'SWA3012': '#00ff9d',   # green
    # add more here if needed
}
DEFAULT_COLOURS = ['#ff4488', '#bf55ff', '#ffee44']

def hex_rgba(hex_color, alpha=1.0):
    """Convert #rrggbb to rgba(r,g,b,alpha) for Plotly."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'



def get_colour(cs, idx):
    return COLOURS.get(cs, DEFAULT_COLOURS[idx % len(DEFAULT_COLOURS)])

# ═══════════════════════════════════════════════════════════════
#  2. MESA AGENT & MODEL
# ═══════════════════════════════════════════════════════════════
class AircraftAgent(Agent):
    def __init__(self, model, callsign: str, trajectory: pd.DataFrame):
        try:
            super().__init__(model)
        except TypeError:
            super().__init__(callsign, model)

        self.callsign    = callsign
        self.trajectory  = trajectory.reset_index(drop=True)
        self.step_index  = 0
        row              = self.trajectory.iloc[0]
        self.lat, self.lon, self.alt, self.speed = row.lat, row.lon, row.Altitude, row.Speed
        self.history_lat = [self.lat]
        self.history_lon = [self.lon]

    def step(self):
        if self.step_index < len(self.trajectory) - 1:
            self.step_index += 1
            row = self.trajectory.iloc[self.step_index]
            self.lat, self.lon = row.lat, row.lon
            self.alt,  self.speed = row.Altitude, row.Speed
            self.history_lat.append(self.lat)
            self.history_lon.append(self.lon)


class AirTrafficModel(Model):
    def __init__(self, aircraft_data: dict):
        super().__init__()
        self.agents_list = []

        # Build scheduler upfront — works on ALL Mesa versions
        try:
            from mesa.time import BaseScheduler
            self.schedule = BaseScheduler(self)
            self._use_schedule = True
        except Exception:
            self._use_schedule = False

        for cs, df in aircraft_data.items():
            agent = AircraftAgent(self, cs, df)
            self.agents_list.append(agent)
            if self._use_schedule:
                self.schedule.add(agent)

    def step(self):
        if self._use_schedule:
            self.schedule.step()
        else:
            # Fallback: step each agent manually
            for agent in self.agents_list:
                agent.step()


# Run simulation
model    = AirTrafficModel(aircraft_data)
max_rows = max(len(df) for df in aircraft_data.values())
print(f"Running simulation — {max_rows} steps, {len(model.agents_list)} agents...")
for _ in range(max_rows - 1):
    model.step()
print("Simulation complete.")

# ═══════════════════════════════════════════════════════════════
#  3. COMPUTE STATS PER AIRCRAFT
# ═══════════════════════════════════════════════════════════════
def haversine_nm(df):
    R = 3440
    lat1 = np.radians(df['lat'].iloc[0]);  lon1 = np.radians(df['lon'].iloc[0])
    lat2 = np.radians(df['lat'].iloc[-1]); lon2 = np.radians(df['lon'].iloc[-1])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return int(2 * R * np.arcsin(np.sqrt(a)))

stats = {}
for cs, df in aircraft_data.items():
    dur_s   = (df['UTC'].iloc[-1] - df['UTC'].iloc[0]).total_seconds()
    h, m    = divmod(int(dur_s / 60), 60)
    stats[cs] = {
        'max_alt':   int(df['Altitude'].max()),
        'avg_speed': int(df['Speed'].mean()),
        'max_speed': int(df['Speed'].max()),
        'duration':  f"{h}h {m:02d}m",
        'dist_nm':   haversine_nm(df),
        'points':    len(df),
        'dep_time':  df['UTC'].iloc[0].strftime('%H:%M UTC'),
        'arr_time':  df['UTC'].iloc[-1].strftime('%H:%M UTC'),
        'dep_coord': f"{df['lat'].iloc[0]:.2f}°N  {abs(df['lon'].iloc[0]):.2f}°W",
        'arr_coord': f"{df['lat'].iloc[-1]:.2f}°N  {abs(df['lon'].iloc[-1]):.2f}°W",
        'colour':    get_colour(cs, list(aircraft_data.keys()).index(cs)),
    }

# ═══════════════════════════════════════════════════════════════
#  4. BUILD PLOTLY FIGURE
# ═══════════════════════════════════════════════════════════════
fig = go.Figure()

all_lats = raw['lat']; all_lons = raw['lon']
mid_lat  = (all_lats.max() + all_lats.min()) / 2
mid_lon  = (all_lons.max() + all_lons.min()) / 2

for idx, (cs, df) in enumerate(aircraft_data.items()):
    colour = get_colour(cs, idx)
    colour_dim = hex_rgba(colour, 0.33)

    # ── path line ──
    fig.add_trace(go.Scattergeo(
        lat=df['lat'], lon=df['lon'],
        mode='lines',
        line=dict(width=1.5, color=hex_rgba(colour, 0.27)),
        hoverinfo='skip', showlegend=False,
    ))

    # ── dots coloured by altitude ──
    fig.add_trace(go.Scattergeo(
        lat=df['lat'], lon=df['lon'],
        mode='markers',
        marker=dict(
            size=4,
            color=df['Altitude'],
            colorscale=[
                [0.0,  '#0a0f2e'],
                [0.3,  hex_rgba(colour, 0.67)],
                [0.7,  colour],
                [1.0,  '#ffffff'],
            ],
            cmin=0, cmax=40000,
            showscale=False,
            opacity=0.9,
        ),
        text=[
            f"<b>{cs}</b><br>"
            f"──────────────<br>"
            f"⏱  {r.UTC.strftime('%H:%M:%S')} UTC<br>"
            f"📍 {r.lat:.4f}°N  {abs(r.lon):.4f}°W<br>"
            f"✈  {int(r.Altitude):,} ft<br>"
            f"💨 {int(r.Speed)} kts<br>"
            f"🧭 {int(r.Direction)}°"
            for _, r in df.iterrows()
        ],
        hoverinfo='text',
        hoverlabel=dict(
            bgcolor='#050c1c',
            bordercolor=colour,
            font=dict(color='#c8eaf5', size=12, family='Courier New'),
        ),
        name=cs,
        showlegend=False,
    ))

    # ── departure marker ──
    fig.add_trace(go.Scattergeo(
        lat=[df['lat'].iloc[0]], lon=[df['lon'].iloc[0]],
        mode='markers',
        marker=dict(size=14, color=colour, symbol='circle',
                    line=dict(color='white', width=2)),
        hovertext=f"<b>DEPARTURE — {cs}</b><br>{stats[cs]['dep_coord']}<br>{stats[cs]['dep_time']}",
        hoverinfo='text',
        hoverlabel=dict(bgcolor='#050c1c', bordercolor=colour,
                        font=dict(color=colour, size=12, family='Courier New')),
        showlegend=False,
    ))

    # ── arrival marker ──
    fig.add_trace(go.Scattergeo(
        lat=[df['lat'].iloc[-1]], lon=[df['lon'].iloc[-1]],
        mode='markers',
        marker=dict(size=14, color=colour, symbol='square',
                    line=dict(color='white', width=1.5)),
        hovertext=f"<b>ARRIVAL — {cs}</b><br>{stats[cs]['arr_coord']}<br>{stats[cs]['arr_time']}",
        hoverinfo='text',
        hoverlabel=dict(bgcolor='#050c1c', bordercolor=colour,
                        font=dict(color=colour, size=12, family='Courier New')),
        showlegend=False,
    ))

# ── geo map style ──
fig.update_geos(
    projection=dict(
        type='orthographic',
        rotation=dict(lon=mid_lon, lat=mid_lat),
    ),
    center=dict(lat=mid_lat, lon=mid_lon),
    showland=True,       landcolor='#0d1b2a',
    showocean=True,      oceancolor='#060d1a',
    showlakes=True,      lakecolor='#081422',
    showcountries=True,  countrycolor='rgba(60,100,160,0.5)',  countrywidth=0.6,
    showcoastlines=True, coastlinecolor='rgba(0,140,200,0.7)', coastlinewidth=1,
    showsubunits=True,   subunitcolor='rgba(40,80,130,0.4)',   subunitwidth=0.5,
    showrivers=False,
    lonaxis=dict(range=[all_lons.min() - 4, all_lons.max() + 4]),
    lataxis=dict(range=[all_lats.min() - 4, all_lats.max() + 4]),
    bgcolor='#030810',
)
fig.update_layout(
    paper_bgcolor='#030810',
    geo_bgcolor='#030810',
    margin=dict(l=0, r=0, t=0, b=0),
    height=900, autosize=True,
    showlegend=False,
)

plotly_div = fig.to_html(full_html=False, include_plotlyjs='cdn', config={
    'displaylogo': False,
    'scrollZoom': True,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d'],
})

# ═══════════════════════════════════════════════════════════════
#  5. BUILD HTML DASHBOARD
# ═══════════════════════════════════════════════════════════════

# generate per-aircraft stat cards
def stat_cards_html():
    cards = []
    for cs, s in stats.items():
        c = s['colour']
        cards.append(f"""
        <div class="ac-card" style="--ac-color:{c};">
          <div class="ac-card-header">
            <span class="ac-dot" style="background:{c}; box-shadow:0 0 8px {c};"></span>
            <span class="ac-callsign">{cs}</span>
          </div>
          <div class="ac-rows">
            <div class="ac-row"><span class="ac-lbl">MAX ALT</span><span class="ac-val" style="color:{c};">{s['max_alt']:,} ft</span></div>
            <div class="ac-row"><span class="ac-lbl">AVG SPD</span><span class="ac-val">{s['avg_speed']} kts</span></div>
            <div class="ac-row"><span class="ac-lbl">DISTANCE</span><span class="ac-val">{s['dist_nm']} nm</span></div>
            <div class="ac-row"><span class="ac-lbl">DURATION</span><span class="ac-val">{s['duration']}</span></div>
            <div class="ac-row"><span class="ac-lbl">POINTS</span><span class="ac-val">{s['points']}</span></div>
          </div>
          <div class="ac-route">
            <span class="ac-time" style="color:{c};">{s['dep_time']}</span>
            <span class="ac-arrow">——✈——</span>
            <span class="ac-time" style="color:{c};">{s['arr_time']}</span>
          </div>
        </div>""")
    return '\n'.join(cards)

def legend_items_html():
    items = []
    for cs, s in stats.items():
        c = s['colour']
        items.append(f"""
        <div class="leg-item">
          <span class="leg-dot" style="background:{c}; box-shadow:0 0 7px {c};"></span>
          <span class="leg-label">{cs}</span>
          <span class="leg-dist">{s['dist_nm']} nm</span>
        </div>""")
    return '\n'.join(items)

total_points = sum(s['points'] for s in stats.values())
date_str = raw['UTC'].iloc[0].strftime('%d %b %Y')

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width,initial-scale=1.0"/>
  <title>Air Traffic — Multi Aircraft</title>
  <link href="https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@300;400;600;700&display=swap" rel="stylesheet">
  <style>
    *, *::before, *::after {{ box-sizing:border-box; margin:0; padding:0; }}
    :root {{
      --bg:    #030810;
      --panel: rgba(4,12,26,0.93);
      --bord:  rgba(0,180,220,0.2);
      --cyan:  #00c8e0;
      --text:  #c8eaf5;
      --muted: #4a7a92;
      --mono:  'Share Tech Mono', monospace;
      --head:  'Rajdhani', sans-serif;
    }}
    html, body {{ width:100%; height:100%; background:var(--bg); color:var(--text); font-family:var(--head); overflow:hidden; }}

    body::before {{
      content:''; position:fixed; inset:0; pointer-events:none; z-index:999;
      background: repeating-linear-gradient(0deg, transparent, transparent 2px, rgba(0,0,0,0.04) 2px, rgba(0,0,0,0.04) 4px);
    }}

    /* ── HEADER ── */
    header {{
      position:fixed; top:0; left:0; right:0; height:50px; z-index:100;
      display:flex; align-items:center; justify-content:space-between; padding:0 20px;
      background:rgba(3,8,16,0.97); border-bottom:1px solid var(--bord); backdrop-filter:blur(12px);
    }}
    .h-left {{ display:flex; align-items:center; gap:12px; }}
    .h-title {{ font-family:var(--mono); font-size:18px; color:var(--cyan); letter-spacing:3px; text-shadow:0 0 16px rgba(0,200,224,0.5); }}
    .badge {{ font-family:var(--mono); font-size:9px; color:var(--muted); background:rgba(0,180,220,0.07); border:1px solid var(--bord); padding:3px 7px; letter-spacing:2px; }}
    .h-right {{ font-family:var(--mono); font-size:10px; color:var(--muted); letter-spacing:1px; }}
    .live-dot {{ display:inline-block; width:7px; height:7px; border-radius:50%; background:#00ff9d; margin-right:5px; box-shadow:0 0 8px #00ff9d; animation:pulse 1.8s infinite; }}
    @keyframes pulse {{ 0%,100%{{opacity:1;transform:scale(1)}} 50%{{opacity:.4;transform:scale(.7)}} }}

    /* ── MAP ── */
    #map-container {{
      position:fixed; inset:50px 0 0 0; z-index:1;
    }}
    #map-container > div,
    #map-container > div > div,
    #map-container .plotly-graph-div {{ width:100% !important; height:100% !important; }}

    /* ── PANEL BASE ── */
    .panel {{
      position:fixed; z-index:50;
      background:var(--panel); border:1px solid var(--bord);
      backdrop-filter:blur(16px); padding:14px 16px;
      animation:fadein 0.7s ease both;
    }}
    .p-title {{ font-family:var(--mono); font-size:9px; letter-spacing:3px; color:var(--muted); text-transform:uppercase; margin-bottom:10px; padding-bottom:8px; border-bottom:1px solid rgba(0,180,220,0.13); }}
    @keyframes fadein {{ from{{opacity:0;transform:translateY(5px)}} to{{opacity:1;transform:translateY(0)}} }}

    /* ── LEFT PANEL — aircraft cards ── */
    #panel-left {{ top:64px; left:14px; width:220px; animation-delay:.1s; }}

    .ac-card {{
      margin-bottom:12px; padding:10px 10px 8px;
      border:1px solid rgba(var(--ac-color-rgb, 0,200,224), 0.18);
      border-left:3px solid var(--ac-color);
      background:rgba(0,0,0,0.25);
      transition: border-color 0.2s;
    }}
    .ac-card:last-child {{ margin-bottom:0; }}
    .ac-card-header {{ display:flex; align-items:center; gap:8px; margin-bottom:8px; }}
    .ac-dot {{ width:10px; height:10px; border-radius:50%; flex-shrink:0; }}
    .ac-callsign {{ font-family:var(--mono); font-size:13px; color:var(--text); letter-spacing:2px; }}
    .ac-rows {{ display:flex; flex-direction:column; gap:4px; margin-bottom:8px; }}
    .ac-row {{ display:flex; justify-content:space-between; align-items:baseline; }}
    .ac-lbl {{ font-family:var(--mono); font-size:8px; color:var(--muted); letter-spacing:1px; }}
    .ac-val {{ font-family:var(--mono); font-size:11px; color:var(--text); }}
    .ac-route {{ display:flex; justify-content:space-between; align-items:center; padding-top:6px; border-top:1px solid rgba(0,180,220,0.1); }}
    .ac-time {{ font-family:var(--mono); font-size:9px; }}
    .ac-arrow {{ font-family:var(--mono); font-size:9px; color:var(--muted); }}

    /* ── RIGHT PANEL — legend + controls ── */
    #panel-right {{ top:64px; right:14px; width:180px; animation-delay:.15s; }}

    .leg-item {{ display:flex; align-items:center; gap:8px; margin-bottom:8px; }}
    .leg-dot {{ width:10px; height:10px; border-radius:50%; flex-shrink:0; }}
    .leg-label {{ font-family:var(--mono); font-size:11px; color:var(--text); flex:1; letter-spacing:1px; }}
    .leg-dist {{ font-family:var(--mono); font-size:9px; color:var(--muted); }}

    .ctrl {{ font-family:var(--mono); font-size:9px; color:var(--muted); line-height:1.9; margin-top:2px; }}

    /* ── BOTTOM BAR — summary ── */
    #panel-bottom {{
      bottom:14px; left:50%; transform:translateX(-50%);
      width:500px; padding:10px 20px;
      animation-delay:.2s;
    }}
    .bot-row {{ display:flex; justify-content:space-between; align-items:center; }}
    .bot-stat {{ text-align:center; }}
    .bot-val {{ font-family:var(--mono); font-size:16px; color:var(--cyan); text-shadow:0 0 10px rgba(0,200,224,0.4); }}
    .bot-lbl {{ font-family:var(--mono); font-size:8px; color:var(--muted); letter-spacing:2px; margin-top:2px; }}
    .bot-div {{ width:1px; height:30px; background:var(--bord); }}

    /* ── CORNER MARKS ── */
    .corner {{ position:fixed; z-index:60; width:18px; height:18px; }}
    .corner-tl {{ top:56px;  left:8px;   border-top:1px solid rgba(0,180,220,0.3); border-left:1px solid rgba(0,180,220,0.3); }}
    .corner-tr {{ top:56px;  right:8px;  border-top:1px solid rgba(0,180,220,0.3); border-right:1px solid rgba(0,180,220,0.3); }}
    .corner-bl {{ bottom:8px; left:8px;  border-bottom:1px solid rgba(0,180,220,0.3); border-left:1px solid rgba(0,180,220,0.3); }}
    .corner-br {{ bottom:8px; right:8px; border-bottom:1px solid rgba(0,180,220,0.3); border-right:1px solid rgba(0,180,220,0.3); }}
  </style>
</head>
<body>

  <header>
    <div class="h-left">
      <span class="h-title">AIR TRAFFIC</span>
      <span class="badge">MULTI-AIRCRAFT</span>
      <span class="badge">{len(aircraft_data)} AGENTS</span>
      <span class="badge">MESA SIM</span>
    </div>
    <div class="h-right">
      <span class="live-dot"></span>REPLAY &nbsp;|&nbsp; {date_str} &nbsp;|&nbsp; {total_points} DATA POINTS
    </div>
  </header>

  <div id="map-container">{plotly_div}</div>

  <div class="corner corner-tl"></div>
  <div class="corner corner-tr"></div>
  <div class="corner corner-bl"></div>
  <div class="corner corner-br"></div>

  <!-- LEFT: per-aircraft cards -->
  <div class="panel" id="panel-left">
    <div class="p-title">▸ Aircraft — {len(aircraft_data)} Agents</div>
    {stat_cards_html()}
  </div>

  <!-- RIGHT: legend + controls -->
  <div class="panel" id="panel-right">
    <div class="p-title">▸ Legend</div>
    {legend_items_html()}
    <div style="margin-top:12px; padding-top:10px; border-top:1px solid rgba(0,180,220,0.12);">
      <div class="p-title" style="margin-bottom:4px;">▸ Controls</div>
      <div class="ctrl">DRAG &nbsp; → rotate globe<br>SCROLL → zoom<br>HOVER &nbsp; → point details<br>● = Departure<br>■ = Arrival</div>
    </div>
  </div>

  <!-- BOTTOM: fleet summary -->
  <div class="panel" id="panel-bottom">
    <div class="bot-row">
      <div class="bot-stat">
        <div class="bot-val">{len(aircraft_data)}</div>
        <div class="bot-lbl">AIRCRAFT</div>
      </div>
      <div class="bot-div"></div>
      <div class="bot-stat">
        <div class="bot-val">{total_points:,}</div>
        <div class="bot-lbl">DATA POINTS</div>
      </div>
      <div class="bot-div"></div>
      <div class="bot-stat">
        <div class="bot-val">{max(s['max_alt'] for s in stats.values()):,} ft</div>
        <div class="bot-lbl">FLEET MAX ALT</div>
      </div>
      <div class="bot-div"></div>
      <div class="bot-stat">
        <div class="bot-val">{sum(s['dist_nm'] for s in stats.values()):,} nm</div>
        <div class="bot-lbl">TOTAL DISTANCE</div>
      </div>
      <div class="bot-div"></div>
      <div class="bot-stat">
        <div class="bot-val">{int(np.mean([s['avg_speed'] for s in stats.values()]))} kts</div>
        <div class="bot-lbl">FLEET AVG SPD</div>
      </div>
    </div>
  </div>

</body>
<script>
  window.addEventListener('load', function() {{
    setTimeout(function() {{
      document.querySelectorAll('.plotly-graph-div').forEach(function(p) {{
        p.style.width = '100%'; p.style.height = '100%';
        if (window.Plotly) Plotly.relayout(p, {{autosize: true}});
      }});
    }}, 300);
  }});
  window.addEventListener('resize', function() {{
    document.querySelectorAll('.plotly-graph-div').forEach(function(p) {{
      if (window.Plotly) Plotly.relayout(p, {{autosize: true}});
    }});
  }});
</script>
</html>"""

out = 'multi_aircraft_dashboard.html'
with open(out, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✈  Saved → {out}")
webbrowser.open('file://' + os.path.abspath(out))

Loaded 3 aircraft: ['AAL1189', 'NKS2061', 'SWA3012']
Running simulation — 877 steps, 3 agents...
Simulation complete.

✈  Saved → multi_aircraft_dashboard.html


True